# Plot KSG MI Results — shendure / SCVI only

Reads `ksg_mutual_information.txt` files written by
`2026-04-20_10-49_compute_ksg_scaling_curves.py` for every
(size, quality) combination on **shendure** with **SCVI**,
checks completeness, and plots MI vs quality.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from itertools import product
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

DATA_ROOT = Path('/home/igor/noise_scaling/data')

ALGOS = ['SCVI']

# shendure-only
EXPECTED = {
    'shendure': {
        'sizes': [100, 359, 1291, 4641, 16681, 59948, 215443, 774263, 2782559, 10000000],
        'qualities': [0.004, 0.0073875, 0.0136438, 0.0251984, 0.0465384, 0.0859506, 0.1587401, 0.2931733, 0.5414548, 1.0],
        'signal': 'author_day',
    },
}

In [ ]:
def _read_mi(path):
    try:
        with open(path) as f:
            return float(f.read().strip())
    except Exception:
        return np.nan

expected_rows = []
for ds, cfg in EXPECTED.items():
    sig = cfg['signal']
    for sz, q, algo in product(cfg['sizes'], cfg['qualities'], ALGOS):
        suffix = '_geneformer' if algo == 'Geneformer' else ''
        stem = f'Y_{sig}_{q}{suffix}'
        mi_path = (DATA_ROOT / ds / str(sz) / str(q) / 'results' / algo / 'model'
                   / 'MI' / 'ksg' / stem / 'ksg_mutual_information.txt')
        expected_rows.append({
            'dataset': ds, 'size': sz, 'quality': q, 'algorithm': algo,
            'signal': sig, 'path': str(mi_path),
        })

df_expected = pd.DataFrame(expected_rows)
print(f'Total expected: {len(df_expected)}')

with ThreadPoolExecutor(max_workers=128) as pool:
    df_expected['exists'] = list(tqdm(pool.map(os.path.exists, df_expected['path']),
                                      total=len(df_expected), desc='Scanning'))

df_found = df_expected[df_expected['exists']].copy()
print(f'Found: {len(df_found)}, Missing: {len(df_expected) - len(df_found)}')

with ThreadPoolExecutor(max_workers=128) as pool:
    df_found['mi_value'] = list(tqdm(pool.map(_read_mi, df_found['path']),
                                     total=len(df_found), desc='Reading MI'))

out_cols = ['dataset', 'size', 'quality', 'algorithm', 'signal', 'mi_value']
df_results = df_found[out_cols].sort_values(out_cols[:-1]).reset_index(drop=True)
print(f'Collected {len(df_results)} KSG MI values (NaN: {df_results["mi_value"].isna().sum()})')

In [ ]:
pivot = df_expected.groupby(['dataset', 'algorithm'])['exists'].agg(['sum', 'count'])
pivot.columns = ['found', 'expected']
pivot['missing'] = pivot['expected'] - pivot['found']
pivot = pivot.astype(int)
pivot_wide = pivot.unstack('algorithm', fill_value=0).sort_index()
display(pivot_wide)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

ALGO_ORDER = ['SCVI']

def _fmt_size(sz):
    """Human-readable size label: 100, 1k, 10k, 1M, 10M, etc."""
    if sz >= 1_000_000:
        return f'{sz/1_000_000:.0f}M' if sz % 1_000_000 == 0 else f'{sz/1_000_000:.1f}M'
    if sz >= 1_000:
        return f'{sz/1_000:.0f}k' if sz % 1_000 == 0 else f'{sz/1_000:.1f}k'
    return str(sz)

def plot_mi_vs_quality(df):
    """Plot MI vs quality: rows=(dataset, signal), cols=algorithm, curves=size.
    Per-dataset color norm and a per-row legend rendered on the right \u2014 built
    from handles collected across every column so the legend always renders
    even if the rightmost algo has no data for that row."""
    all_ds_sig = sorted(set(zip(df['dataset'], df['signal'])))
    algorithms = [a for a in ALGO_ORDER if a in df['algorithm'].unique()]
    n_rows = len(all_ds_sig)
    n_cols = len(algorithms)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols + 1.6, 3.5 * n_rows + 0.8),
                             squeeze=False, sharex=False)

    # Share y-axis within each row
    for i in range(n_rows):
        for j in range(1, n_cols):
            axes[i, j].sharey(axes[i, 0])

    cmap = plt.cm.viridis

    # Per-dataset size norms
    ds_norms = {}
    for ds, _ in all_ds_sig:
        sizes = sorted(df[df['dataset'] == ds]['size'].unique())
        if sizes:
            ds_norms[ds] = mcolors.LogNorm(vmin=min(sizes), vmax=max(sizes))

    for i, (ds, sig) in enumerate(all_ds_sig):
        norm = ds_norms.get(ds)
        # Gather one handle per size for this row, picking whichever column
        # supplies it first (sorted by size so labels stay ordered).
        row_handles: dict[float, tuple] = {}
        for j, algo in enumerate(algorithms):
            ax = axes[i, j]
            sub = df[(df['dataset'] == ds) & (df['algorithm'] == algo) & (df['signal'] == sig)]

            if sub.empty:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, color='gray', fontsize=10)
                ax.set_xscale('log')
            else:
                for sz in sorted(sub['size'].unique()):
                    color = cmap(norm(sz))
                    sz_df = sub[sub['size'] == sz]
                    agg = sz_df.groupby('quality')['mi_value'].agg(['mean', 'std', 'count']).reset_index()
                    agg = agg.sort_values('quality')
                    agg['sem'] = (agg['std'] / np.sqrt(agg['count'])).fillna(0)
                    errbar = ax.errorbar(agg['quality'], agg['mean'], yerr=agg['sem'],
                                          marker='o', markersize=3, linewidth=1, capsize=2,
                                          color=color, alpha=0.85, label=_fmt_size(sz))
                    if sz not in row_handles:
                        row_handles[sz] = (errbar[0], _fmt_size(sz))
                ax.set_xscale('log')

            if i == n_rows - 1:
                ax.set_xlabel('Quality (downsampling ratio)', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'{ds} \u2014 KSG MI ({sig})', fontsize=10)
            if i == 0:
                ax.set_title(algo, fontsize=12, fontweight='bold')
            ax.tick_params(labelsize=8)
            if j > 0:
                ax.tick_params(labelleft=False)

        # Per-row legend on the rightmost axis (always renders if the row has data).
        if row_handles:
            sizes_sorted = sorted(row_handles.keys())
            handles = [row_handles[s][0] for s in sizes_sorted]
            labels  = [row_handles[s][1] for s in sizes_sorted]
            axes[i, n_cols - 1].legend(
                handles, labels, title='# cells', fontsize=6, title_fontsize=7,
                loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0,
                frameon=True, framealpha=0.8,
            )

    fig.suptitle('KSG MI vs Quality (from disk)',
                 fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0.01, 0.92, 0.96])
    plt.show()
    return fig

fig_quality = plot_mi_vs_quality(df_results)